
##  什么是 Memory？

在 LangChain 中，Memory 用于在多轮对话中保存和管理上下文信息。
它会自动记录用户输入 (input) 与模型输出 (output)，
并在下一轮调用时自动把历史对话注入到 Prompt 中，让 LLM 有“记忆”能力。



###  常见 Memory 类型对比

| Memory 类型 | 特点 | 适用场景 |
|--------------|------|-----------|
| **ConversationBufferMemory** | 保存完整的对话历史（按字符串拼接） | 简单聊天场景 |
| **ConversationBufferWindowMemory** | 只保留最近 N 轮对话 | 控制上下文长度，节省 token |
| **ConversationSummaryMemory** | 用 LLM 总结历史对话，再保存摘要 | 长对话（节省上下文空间） |
| **ConversationTokenBufferMemory** | 按 token 数量控制记忆范围 | 精确控制上下文长度 |
| **VectorStoreRetrieverMemory** | 将历史内容向量化存入向量库（如 Milvus/FAISS），支持语义检索 | 长期知识记忆、知识问答 |
| **EntityMemory** | 按“实体（人物/地点/事件）”追踪信息 | 角色扮演、多角色记忆场景 |

###  工作原理（以 ConversationBufferMemory 为例）
保存完整对话，一直往后拼接。


In [ ]:


from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_community.chat_models import ChatOpenAI

llm = ChatOpenAI(model="qwen-plus", temperature=0)
memory = ConversationBufferMemory()  # 保存全部历史记录

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print(conversation.predict(input="你好，我叫张三"))
print(conversation.predict(input="我刚才说我叫什么名字？"))

输出：

> Memory 会自动注入：
Human: 你好，我叫张三
AI: 你好张三！
Human: 我刚才说我叫什么名字？
AI: 你说你叫张三。
> 
###  使用 Memory 的好处
| 功能 | 效果 |
|------|--------|
| 自动保存对话历史 | 模型可以回忆前文，不需手动拼接 prompt |
| 控制上下文长度 | 使用 Window / Token Memory 可避免超长上下文 |
| 支持语义检索 | Vector Memory 可实现长期记忆或知识库召回 |
| 提升多轮体验 | 让 Agent 或聊天机器人更像“有记忆的人” |

###  在 Agent 中使用 Memory

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history")

agent = initialize_agent(
    tools=[],
    llm=llm,
    agent_type=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

agent.run("你好")
agent.run("我刚才说了什么？")

ConversationBufferWindowMemory 示例：
只保留最近 N 轮，老对话会被丢弃。


In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain_community.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# 只记录最近 2 轮对话
memory = ConversationBufferWindowMemory(
    k=2,               # window size
    return_messages=True
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

conversation.run("我叫小张，今年28岁，住在上海，你记住我。")
conversation.run("顺便记一下：我喜欢 Python。")
conversation.run("再记一下：我喜欢打游戏。")

print("=" * 40)
print(conversation.run("我之前说我住在哪里？"))
print("=" * 40)
print("window 内的对话：")
for m in memory.chat_memory.messages:
    print(m)
#当你把 k 设得很小，早期“我住在上海”的那轮可能会被丢掉，模型就答不上来。


### ConversationSummaryMemory

用 LLM 把历史对话“压缩成摘要”，再一起带给模型。

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationSummaryMemory
from langchain_community.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# 用一个额外的 llm 做 summarizer（也可和主 llm 一样）
summary_llm = ChatOpenAI(model="gpt-4o-mini")

memory = ConversationSummaryMemory(
    llm=summary_llm,
    return_messages=True
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

conversation.run("我叫小张，今年28岁，住在上海，你记住我。")
conversation.run("我最近在学深度学习。")
conversation.run("周末一般打游戏或者出去爬山。")

print("=" * 40)
print(conversation.run("你还记得我是做什么的吗？我住哪？"))
print("=" * 40)
print("当前 summary：")
print(memory.moving_summary_buffer)

### 4. ConversationTokenBufferMemory

按 token 数量 控制记忆长度，而不是轮数。

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationTokenBufferMemory
from langchain_community.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# 这里需要给一个 encoding 或者直接让 LangChain 自动推（部分版本已支持）
memory = ConversationTokenBufferMemory(
    llm=llm,                 # 用于估算 token 数
    max_token_limit=200,     # 总记忆不超过 200 tokens
    return_messages=True,
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

for i in range(10):
    conversation.run(f"第{i}轮：我在说一些废话用来占 token，你记一下。")

print("=" * 40)
print(conversation.run("你现在还能记得最早我说了什么吗？"))
print("=" * 40)
print("当前记忆（受 token 限制）：")
print(memory.buffer)

### 5. VectorStoreRetrieverMemory（Milvus / FAISS）

历史内容 + 知识统一丢向量库，靠语义检索找回来。
下面用 FAISS 做 demo，换 Milvus 只要改 vectorstore 初始化部分。

In [ ]:
import faiss
from langchain_community.chat_models import ChatOpenAI,OpenAIEmbeddings

from langchain.vectorstores import FAISS
from langchain.memory import VectorStoreRetrieverMemory
from langchain.chains import ConversationalRetrievalChain

# 1. 向量库
emb = OpenAIEmbeddings(model="text-embedding-3-small")
# 初始化一个空的 FAISS
vector_store = FAISS.from_texts(
    texts=["这里是一个占位文本，避免空索引"], 
    embedding=emb
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

memory = VectorStoreRetrieverMemory(retriever=retriever)

# 2. 写一个简单的“对话 + 检索”链
llm = ChatOpenAI(model="gpt-4o-mini")

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# 往 memory 里“存”几条对话 / 知识（其实就是 add_documents 到向量库）
memory.save_context(
    {"input": "我叫小张，住在上海浦东新区。"},
    {"output": "好的，我已经记住你在上海浦东。"}
)
memory.save_context(
    {"input": "我最常用的语言是 Python 和 Java。"},
    {"output": "我已经记住你是 Python / Java 开发。"}
)

print("=" * 40)
result = qa(
    {"question": "我住在哪里？我是写什么语言的？"},
)
print(result["answer"])
print("=" * 40)
print("召回到的历史片段：")
for doc in result["source_documents"]:
    print("-", doc.page_content)

### 6. EntityMemory

按“实体”组织记忆：人名 / 地点 / 公司等，适合长期多角色扮演。

In [ ]:
from langchain.memory import ConversationEntityMemory
from langchain.chains import ConversationChain
from langchain_community.chat_models import ChatOpenAI,OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4o-mini")

memory = ConversationEntityMemory(
    llm=llm,
    k=3,                      # 每个实体最多记多少条描述
)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

conversation.run("我叫小张，住在上海浦东，在一家互联网公司做后端。")
conversation.run("我女朋友叫小李，在公安系统上班，现在在广西。")
conversation.run("我爸妈都在钦州老家，养了两只猫。")

print("=" * 40)
print(conversation.run("请简单介绍一下我女朋友是做什么的？"))
print("=" * 40)
print("当前实体记忆：")
print(memory.entity_store.store)   # 按实体划分的信息


一句话总结：

Memory 就是 LangChain 让模型“记住上下文”的机制。 可以选择不同类型的记忆方式（Buffer、Window、Summary、Vector）来平衡 记忆能力 vs 成本。
